# Supplementary Code — Brazilian Ethanol Road-Freight Clustering

This notebook is part of the supplementary code package for the manuscript **“Unsupervised Learning for Spatiotemporal Pattern Discovery in Brazilian Ethanol Road Freight Prices: Implications for Sustainable Logistics.”**

**Data are not distributed with this code package.** The private ESALQ-Log freight quotations are third-party data subject to academic-use restrictions. Public contextual data must be downloaded from their original providers. See `../data/README.md` and the package-level `README.md`.

The publication copy contains code and methodological documentation only. Cell outputs were intentionally removed to avoid redistributing restricted data or derivative tables inadvertently.


# Análise Exploratória de Variáveis Externas
## Produção, infraestrutura e demanda relacionadas ao mercado de etanol

Objetivo:
Avaliar cinco bases externas candidatas à caracterização e/ou ampliação
do clustering dos preços de frete rodoviário de álcool:

1. Produção municipal de cana-de-açúcar
2. Produção estadual de etanol
3. Localização e capacidade das usinas
4. Localização das distribuidoras
5. Localização dos postos de combustíveis

In [ ]:
# INPUT CONFIGURATION — public contextual datasets are not distributed
from pathlib import Path
import os

DATA_DIR = Path(os.environ.get("ETHANOL_CODE_DATA_DIR", "../data")).resolve()
EXPECTED_INPUTS = [
    "prod_cana_municipal.csv",
    "prod_etanol_estadual.csv",
    "loc_usinas_etanol.csv",
    "distribuidoras.csv",
    "postos.csv",
]
missing = [name for name in EXPECTED_INPUTS if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        "Missing contextual input file(s): " + ", ".join(missing) +
        ". See ../data/README.md for source and schema notes."
    )
print("Contextual input directory configured:", DATA_DIR)


In [ ]:
# INPUT CONFIGURATION — public contextual datasets are not distributed
from pathlib import Path
import os

DATA_DIR = Path(os.environ.get("ETHANOL_CODE_DATA_DIR", "../data")).resolve()
EXPECTED_INPUTS = [
    "prod_cana_municipal.csv",
    "prod_etanol_estadual.csv",
    "loc_usinas_etanol.csv",
    "distribuidoras.csv",
    "postos.csv",
]
missing = [name for name in EXPECTED_INPUTS if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        "Missing contextual input file(s): " + ", ".join(missing) +
        ". See ../data/README.md for source and schema notes."
    )
print("Contextual input directory configured:", DATA_DIR)


Input files are resolved through `DATA_DIR`; no interactive upload is used in the publication version.

In [ ]:
# BLOCO 3 — IDENTIFICAÇÃO DOS ARQUIVOS

ARQ_CANA = DATA_DIR / "prod_cana_municipal.csv"

ARQ_ETANOL = DATA_DIR / "prod_etanol_estadual.csv"

ARQ_USINAS = DATA_DIR / "loc_usinas_etanol.csv"

ARQ_DISTRIBUIDORAS = DATA_DIR / "distribuidoras.csv"

ARQ_POSTOS = DATA_DIR / "postos.csv"

In [ ]:
# BLOCO 4 — LEITURA ROBUSTA DOS CSV
# Tenta automaticamente diferentes codificações

def ler_csv(caminho):

    codificacoes = [
        "utf-8-sig",
        "utf-8",
        "cp1252",
        "latin1"
    ]

    for encoding in codificacoes:

        try:

            df = pd.read_csv(
                caminho,
                sep=None,
                engine="python",
                encoding=encoding
            )

            print(
                f"✓ {caminho} lido com sucesso "
                f"usando encoding: {encoding}"
            )

            return df

        except UnicodeDecodeError:

            continue

    raise ValueError(
        f"Não foi possível identificar a codificação de {caminho}"
    )

# CARREGAR AS CINCO BASES

df_cana = ler_csv(
    ARQ_CANA
)

df_etanol = ler_csv(
    ARQ_ETANOL
)

df_usinas = ler_csv(
    ARQ_USINAS
)

df_distribuidoras = ler_csv(
    ARQ_DISTRIBUIDORAS
)

df_postos = ler_csv(
    ARQ_POSTOS
)

print(
    "\nTodas as bases foram carregadas."
)

In [ ]:
# DIAGNÓSTICO DA BASE DE CANA

print("FORMATO DA BASE:")
print(df_cana.shape)

print("\nCOLUNAS LIDAS:")
print(df_cana.columns.tolist())

print("\nPRIMEIRAS 10 LINHAS:")
display(df_cana.head(10))

In [ ]:
# BLOCO 5 — NORMALIZAÇÃO DOS NOMES DAS COLUNAS

def normalizar_coluna(nome):

    nome = str(nome)

    nome = (
        unicodedata
        .normalize("NFKD", nome)
        .encode("ascii", "ignore")
        .decode("utf-8")
    )

    nome = nome.lower().strip()

    nome = re.sub(
        r"[^a-z0-9]+",
        "_",
        nome
    )

    return nome.strip("_")


bases = {
    "Cana": df_cana,
    "Etanol": df_etanol,
    "Usinas": df_usinas,
    "Distribuidoras": df_distribuidoras,
    "Postos": df_postos
}


for nome, base in bases.items():

    base.columns = [
        normalizar_coluna(col)
        for col in base.columns
    ]

    print(
        "\n==============================="
    )

    print(nome)

    print(
        base.columns.tolist()
    )

In [ ]:
# BLOCO 6 — DIAGNÓSTICO GERAL

resumo_bases = []

for nome, base in bases.items():

    total = base.size

    faltantes = (
        base.isna()
        .sum()
        .sum()
    )

    resumo_bases.append({

        "Base":
            nome,

        "Linhas":
            base.shape[0],

        "Colunas":
            base.shape[1],

        "Duplicatas":
            base.duplicated().sum(),

        "Valores_faltantes":
            faltantes,

        "Pct_faltante":
            faltantes /
            total * 100
    })


resumo_bases = pd.DataFrame(
    resumo_bases
)

display(
    resumo_bases.round(2)
)

In [ ]:
# BLOCO 7 — FALTANTES POR COLUNA

for nome, base in bases.items():

    print(
        "\n================================"
    )

    print(nome.upper())

    print(
        "================================"
    )

    faltantes = pd.DataFrame({

        "Quantidade":
            base.isna().sum(),

        "Percentual":
            base.isna().mean() * 100

    })

    display(
        faltantes[
            faltantes["Quantidade"] > 0
        ]
        .sort_values(
            "Percentual",
            ascending=False
        )
        .round(2)
    )

In [ ]:
# BLOCO 8 — CONVERSÃO NUMÉRICA ROBUSTA

def converter_numerico(serie):

    if pd.api.types.is_numeric_dtype(
        serie
    ):
        return serie

    s = (
        serie
        .astype(str)
        .str.strip()
    )

    resultado = pd.Series(
        np.nan,
        index=s.index,
        dtype=float
    )

    possui_virgula = (
        s.str.contains(
            ",",
            regex=False
        )
    )

    resultado[
        possui_virgula
    ] = pd.to_numeric(

        s[
            possui_virgula
        ]
        .str.replace(
            ".",
            "",
            regex=False
        )
        .str.replace(
            ",",
            ".",
            regex=False
        ),

        errors="coerce"
    )

    resultado[
        ~possui_virgula
    ] = pd.to_numeric(

        s[
            ~possui_virgula
        ],

        errors="coerce"
    )

    return resultado

In [ ]:
# BLOCO 9 — FUNÇÃO DE OUTLIERS

def tabela_outliers(
    df,
    variaveis
):

    resultados = []

    for var in variaveis:

        valores = (
            df[var]
            .dropna()
        )

        if len(valores) == 0:
            continue

        q1 = valores.quantile(0.25)

        q3 = valores.quantile(0.75)

        iqr = q3 - q1

        inferior = (
            q1 - 1.5 * iqr
        )

        superior = (
            q3 + 1.5 * iqr
        )

        n = (
            (
                valores < inferior
            )
            |
            (
                valores > superior
            )
        ).sum()

        resultados.append({

            "Variavel":
                var,

            "Limite_inferior":
                inferior,

            "Limite_superior":
                superior,

            "Quantidade_outliers":
                n
        })

    return pd.DataFrame(
        resultados
    )

In [ ]:
# BLOCO 10 — NORMALIDADE

def tabela_normalidade(
    df,
    variaveis
):

    resultados = []

    for var in variaveis:

        valores = (
            df[var]
            .dropna()
        )

        if len(valores) < 3:
            continue

        # Evita problema do Shapiro
        # em bases muito grandes
        if len(valores) > 5000:

            valores = valores.sample(
                5000,
                random_state=42
            )

        estatistica, p = (
            shapiro(valores)
        )

        resultados.append({

            "Variavel":
                var,

            "Estatistica":
                estatistica,

            "P_valor":
                p,

            "Semelhante_normal":
                (
                    "Sim"
                    if p > 0.05
                    else "Não"
                )
        })

    return pd.DataFrame(
        resultados
    )

In [ ]:
# BLOCO 11 — COEFICIENTE DE VARIAÇÃO

def tabela_variacao(
    df,
    variaveis
):

    tabela = pd.DataFrame({

        "Media":
            df[
                variaveis
            ].mean(),

        "Desvio_padrao":
            df[
                variaveis
            ].std()
    })

    tabela["CV_pct"] = (
        tabela[
            "Desvio_padrao"
        ]
        /
        tabela[
            "Media"
        ].abs()
    ) * 100

    return tabela.sort_values(
        "CV_pct",
        ascending=False
    )

#1. PRODUÇÃO MUNICIPAL DE CANA

In [ ]:
# BLOCO 12 — CONFIGURAÇÃO: PRODUÇÃO MUNICIPAL DE CANA

COL_CANA_MUNICIPIO = "nome_mun"
COL_CANA_UF = "reg_uf"
COL_CANA_CODIGO = "cd_geocodi"

ANOS_CANA = [
    "2021",
    "2022",
    "2023",
    "2024"
]

print("Configuração da base de cana definida.")

In [ ]:
# BLOCO 13 — TRANSFORMAR CANA DE WIDE PARA LONG

cana = df_cana[
    [
        COL_CANA_CODIGO,
        COL_CANA_MUNICIPIO,
        COL_CANA_UF
    ]
    + ANOS_CANA
].copy()


# Renomear colunas de identificação
cana = cana.rename(
    columns={
        COL_CANA_CODIGO:
            "Codigo_IBGE",

        COL_CANA_MUNICIPIO:
            "Municipio",

        COL_CANA_UF:
            "UF"
    }
)


# Transformar anos das colunas em linhas
cana = cana.melt(
    id_vars=[
        "Codigo_IBGE",
        "Municipio",
        "UF"
    ],
    value_vars=ANOS_CANA,
    var_name="Ano",
    value_name="Producao_cana"
)


# Tipos corretos
cana["Ano"] = pd.to_numeric(
    cana["Ano"],
    errors="coerce"
)


cana["Producao_cana"] = pd.to_numeric(
    cana["Producao_cana"],
    errors="coerce"
)


# Código IBGE como texto
cana["Codigo_IBGE"] = (
    cana["Codigo_IBGE"]
    .astype("Int64")
    .astype(str)
)


print(
    "Formato final:",
    cana.shape
)

display(
    cana.head(10)
)

In [ ]:
# BLOCO 14 — QUALIDADE DA BASE DE CANA

print(
    "Registros:",
    len(cana)
)

print(
    "Municípios únicos:",
    cana["Municipio"].nunique()
)

print(
    "UFs:",
    cana["UF"].nunique()
)

print(
    "Período:",
    cana["Ano"].min(),
    "-",
    cana["Ano"].max()
)

print(
    "Valores ausentes:",
    cana["Producao_cana"]
    .isna()
    .sum()
)

print(
    "Produção igual a zero:",
    (
        cana["Producao_cana"] == 0
    ).sum()
)

print(
    "Produção negativa:",
    (
        cana["Producao_cana"] < 0
    ).sum()
)

print(
    "Duplicatas município-ano:",
    cana.duplicated(
        subset=[
            "Codigo_IBGE",
            "Ano"
        ]
    ).sum()
)

In [ ]:
# BLOCO 15 — COBERTURA TEMPORAL DA CANA

cobertura_cana = (
    cana
    .groupby("Ano")
    .agg(

        Municipios_com_registro=(
            "Codigo_IBGE",
            "nunique"
        ),

        Municipios_com_producao_positiva=(
            "Producao_cana",
            lambda x: (x > 0).sum()
        ),

        Valores_ausentes=(
            "Producao_cana",
            lambda x: x.isna().sum()
        ),

        Producao_total=(
            "Producao_cana",
            "sum"
        )
    )
    .reset_index()
)

display(
    cobertura_cana
)

In [ ]:
# BLOCO 16 — PERFIL MUNICIPAL DA PRODUÇÃO DE CANA

cana_municipal = (
    cana
    .groupby(
        [
            "Codigo_IBGE",
            "Municipio",
            "UF"
        ]
    )
    .agg(

        Media_cana=(
            "Producao_cana",
            "mean"
        ),

        Mediana_cana=(
            "Producao_cana",
            "median"
        ),

        Desvio_cana=(
            "Producao_cana",
            "std"
        ),

        Minimo_cana=(
            "Producao_cana",
            "min"
        ),

        Maximo_cana=(
            "Producao_cana",
            "max"
        ),

        Anos_observados=(
            "Ano",
            "nunique"
        )
    )
    .reset_index()
)


cana_municipal[
    "Amplitude_cana"
] = (
    cana_municipal[
        "Maximo_cana"
    ]
    -
    cana_municipal[
        "Minimo_cana"
    ]
)


cana_municipal[
    "CV_cana_pct"
] = np.where(

    cana_municipal[
        "Media_cana"
    ] != 0,

    (
        cana_municipal[
            "Desvio_cana"
        ]
        /
        cana_municipal[
            "Media_cana"
        ]
    ) * 100,

    np.nan
)


display(
    cana_municipal.head(10)
)

In [ ]:
# BLOCO 17 — EDA DA PRODUÇÃO MUNICIPAL DE CANA

variaveis_cana = [
    "Media_cana",
    "Mediana_cana",
    "Desvio_cana",
    "Minimo_cana",
    "Maximo_cana",
    "Amplitude_cana",
    "CV_cana_pct"
]


print(
    "========== OUTLIERS =========="
)

display(
    tabela_outliers(
        cana_municipal,
        variaveis_cana
    ).round(2)
)


print(
    "\n========== VARIAÇÃO RELATIVA =========="
)

display(
    tabela_variacao(
        cana_municipal,
        variaveis_cana
    ).round(2)
)


print(
    "\n========== NORMALIDADE =========="
)

display(
    tabela_normalidade(
        cana_municipal,
        variaveis_cana
    )
)

In [ ]:
# BLOCO 18 — CORRELAÇÃO DAS VARIÁVEIS DA CANA

correlacao_cana = (
    cana_municipal[
        variaveis_cana
    ]
    .corr(
        method="pearson"
    )
)

display(
    correlacao_cana.round(3)
)


plt.figure(
    figsize=(9, 7)
)

sns.heatmap(
    correlacao_cana,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title(
    "Correlação das variáveis de produção municipal de cana"
)

plt.tight_layout()
plt.show()

In [ ]:
# BLOCO 19 — DISTRIBUIÇÕES DA PRODUÇÃO DE CANA

for var in variaveis_cana:

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11, 4)
    )

    sns.histplot(
        data=cana_municipal,
        x=var,
        kde=True,
        ax=axes[0]
    )

    sns.boxplot(
        data=cana_municipal,
        x=var,
        ax=axes[1]
    )

    axes[0].set_title(
        f"Histograma — {var}"
    )

    axes[1].set_title(
        f"Boxplot — {var}"
    )

    plt.tight_layout()
    plt.show()

#2. PRODUÇÃO ESTADUAL DE ETANOL

In [ ]:
# BLOCO 20 — DIAGNÓSTICO DA BASE DE ETANOL

print("FORMATO DA BASE:")
print(df_etanol.shape)

print("\nCOLUNAS DISPONÍVEIS:")
print(df_etanol.columns.tolist())

print("\nPRIMEIRAS 10 LINHAS:")
display(df_etanol.head(10))

In [ ]:
# BLOCO 21 — CONFIGURAÇÃO DA BASE DE ETANOL

# UFs oficiais do Brasil
ufs_validas = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES",
    "GO", "MA", "MT", "MS", "MG", "PA", "PB", "PR",
    "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC",
    "SP", "SE", "TO"
]

anos_etanol = [
    "2021",
    "2022",
    "2023",
    "2024",
    "2025",
    "2026"
]

print("Linhas originais:", len(df_etanol))

print("\nValores encontrados em reg_uf:")
print(df_etanol["reg_uf"].tolist())

In [ ]:
# BLOCO 22 — TRATAMENTO DA PRODUÇÃO ESTADUAL DE ETANOL

etanol = df_etanol[
    df_etanol["reg_uf"].isin(ufs_validas)
].copy()

# Renomear
etanol = etanol.rename(
    columns={
        "reg_uf": "UF"
    }
)

# Transformar:
# UF | 2021 | 2022 | ...
#
# em:
# UF | Ano | Producao_etanol

etanol = etanol.melt(
    id_vars=["UF"],
    value_vars=anos_etanol,
    var_name="Ano",
    value_name="Producao_etanol"
)

# Tipos numéricos
etanol["Ano"] = pd.to_numeric(
    etanol["Ano"],
    errors="coerce"
)

etanol["Producao_etanol"] = pd.to_numeric(
    etanol["Producao_etanol"],
    errors="coerce"
)

print("Formato final:", etanol.shape)

display(
    etanol.head(15)
)

In [ ]:
# BLOCO 23 — QUALIDADE DA PRODUÇÃO ESTADUAL DE ETANOL

print(
    "Registros:",
    len(etanol)
)

print(
    "UFs:",
    etanol["UF"].nunique()
)

print(
    "Período:",
    etanol["Ano"].min(),
    "-",
    etanol["Ano"].max()
)

In [ ]:
# BLOCO 24 — COBERTURA POR ANO

cobertura_etanol = (
    etanol
    .groupby("Ano")
    .agg(

        Estados_com_registro=(
            "UF",
            "nunique"
        ),

        Estados_com_producao_positiva=(
            "Producao_etanol",
            lambda x: (x > 0).sum()
        ),

        Valores_ausentes=(
            "Producao_etanol",
            lambda x: x.isna().sum()
        ),

        Producao_total=(
            "Producao_etanol",
            "sum"
        )
    )
    .reset_index()
)

display(
    cobertura_etanol
)

In [ ]:
plt.figure(
    figsize=(9, 4)
)

sns.lineplot(
    data=cobertura_etanol,
    x="Ano",
    y="Producao_total",
    marker="o"
)

plt.title(
    "Evolução da produção estadual agregada de etanol"
)

plt.xlabel("Ano")
plt.ylabel("Produção total")

plt.tight_layout()
plt.show()

In [ ]:
# BLOCO 25 — PERFIL HISTÓRICO DOS ESTADOS

etanol_estado = (
    etanol
    .groupby("UF")
    .agg(

        Media_etanol=(
            "Producao_etanol",
            "mean"
        ),

        Mediana_etanol=(
            "Producao_etanol",
            "median"
        ),

        Desvio_etanol=(
            "Producao_etanol",
            "std"
        ),

        Minimo_etanol=(
            "Producao_etanol",
            "min"
        ),

        Maximo_etanol=(
            "Producao_etanol",
            "max"
        ),

        Anos_observados=(
            "Ano",
            "nunique"
        )
    )
    .reset_index()
)


etanol_estado[
    "Amplitude_etanol"
] = (
    etanol_estado[
        "Maximo_etanol"
    ]
    -
    etanol_estado[
        "Minimo_etanol"
    ]
)


etanol_estado[
    "CV_etanol_pct"
] = np.where(

    etanol_estado[
        "Media_etanol"
    ] != 0,

    (
        etanol_estado[
            "Desvio_etanol"
        ]
        /
        etanol_estado[
            "Media_etanol"
        ]
    ) * 100,

    np.nan
)


display(
    etanol_estado
    .sort_values(
        "Media_etanol",
        ascending=False
    )
)

In [ ]:
# BLOCO 26 — EDA DA PRODUÇÃO ESTADUAL DE ETANOL

variaveis_etanol = [
    "Media_etanol",
    "Mediana_etanol",
    "Desvio_etanol",
    "Minimo_etanol",
    "Maximo_etanol",
    "Amplitude_etanol",
    "CV_etanol_pct"
]

# 1. OUTLIERS

print(
    "========== OUTLIERS =========="
)

display(
    tabela_outliers(
        etanol_estado,
        variaveis_etanol
    ).round(2)
)

# 2. VARIAÇÃO RELATIVA

print(
    "\n========== VARIAÇÃO RELATIVA =========="
)

display(
    tabela_variacao(
        etanol_estado,
        variaveis_etanol
    ).round(2)
)

# 3. NORMALIDADE

print(
    "\n========== NORMALIDADE =========="
)

display(
    tabela_normalidade(
        etanol_estado,
        variaveis_etanol
    )
)

# 4. CORRELAÇÃO

print(
    "\n========== CORRELAÇÃO =========="
)

corr_etanol = (
    etanol_estado[
        variaveis_etanol
    ]
    .corr(
        method="pearson"
    )
)

display(
    corr_etanol.round(3)
)

In [ ]:
plt.figure(
    figsize=(9, 7)
)

sns.heatmap(
    corr_etanol,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title(
    "Correlação das variáveis de produção estadual de etanol"
)

plt.tight_layout()
plt.show()

In [ ]:
# BLOCO 26.1 — DISTRIBUIÇÕES

for var in variaveis_etanol:

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11, 4)
    )

    sns.histplot(
        data=etanol_estado,
        x=var,
        kde=True,
        ax=axes[0]
    )

    sns.boxplot(
        data=etanol_estado,
        x=var,
        ax=axes[1]
    )

    axes[0].set_title(
        f"Histograma — {var}"
    )

    axes[1].set_title(
        f"Boxplot — {var}"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# BLOCO 26.2 — MAIORES ESTADOS PRODUTORES

top_estados = (
    etanol_estado
    .sort_values(
        "Media_etanol",
        ascending=False
    )
    .head(15)
)

plt.figure(
    figsize=(10, 6)
)

sns.barplot(
    data=top_estados,
    x="Media_etanol",
    y="UF"
)

plt.title(
    "15 estados com maior produção média de etanol"
)

plt.xlabel(
    "Produção média"
)

plt.ylabel(
    "UF"
)

plt.tight_layout()
plt.show()

#3. LOCALIZAÇÃO E CAPACIDADE DAS USINAS

In [ ]:
# BLOCO 27 — DIAGNÓSTICO DA BASE DE USINAS

print("FORMATO:")
print(df_usinas.shape)

print("\nCOLUNAS:")
print(df_usinas.columns.tolist())

print("\nPRIMEIRAS LINHAS:")
display(df_usinas.head(10))

In [ ]:
# BLOCO 28 — DIAGNÓSTICO DAS CATEGORIAS DAS USINAS

print("========== SITUAÇÃO ==========")

display(
    df_usinas["situacao"]
    .value_counts(
        dropna=False
    )
    .to_frame("Quantidade")
)


print("\n========== PRODUTOS ==========")

display(
    df_usinas["produtos"]
    .value_counts(
        dropna=False
    )
    .to_frame("Quantidade")
)


print("\n========== CLASSES DE CAPACIDADE ==========")

display(
    df_usinas["classe_capacidade"]
    .value_counts(
        dropna=False
    )
    .to_frame("Quantidade")
)


print("\n========== UFs ==========")

display(
    df_usinas["reg_uf"]
    .value_counts()
    .to_frame("N_usinas")
)

In [ ]:
# BLOCO 29 — TRATAMENTO DA BASE DE USINAS

usinas = df_usinas.rename(
    columns={
        "reg_uf":
            "UF",

        "nome_usina":
            "Usina",

        "nome_mun":
            "Municipio",

        "latitude":
            "Latitude",

        "longitude":
            "Longitude",

        "produtos":
            "Produtos",

        "situacao":
            "Situacao",

        "classe_capacidade":
            "Classe_capacidade",

        "capacidade_proces_milho":
            "Capacidade_milho",

        "capacidade_proces_anidro":
            "Capacidade_anidro",

        "capacidade_proces_hidratado":
            "Capacidade_hidratado",

        "construcao":
            "Ano_construcao"
    }
).copy()

# Variáveis numéricas

colunas_numericas_usinas = [
    "Latitude",
    "Longitude",
    "Capacidade_milho",
    "Capacidade_anidro",
    "Capacidade_hidratado",
    "Ano_construcao"
]


for coluna in colunas_numericas_usinas:

    usinas[coluna] = pd.to_numeric(
        usinas[coluna],
        errors="coerce"
    )


# Capacidade combinada de processamento de etanol
# IMPORTANTE:
# não chamamos isso de capacidade de moagem.
# É somente a soma das capacidades informadas
# para anidro + hidratado.

usinas["Capacidade_etanol_total"] = (
    usinas["Capacidade_anidro"].fillna(0)
    +
    usinas["Capacidade_hidratado"].fillna(0)
)


print(
    "Formato final:",
    usinas.shape
)

display(
    usinas.head(10)
)

In [ ]:
# BLOCO 30 — QUALIDADE DA BASE DE USINAS

print(
    "Registros:",
    len(usinas)
)

print(
    "Usinas únicas:",
    usinas["Usina"].nunique()
)

print(
    "Municípios:",
    usinas["Municipio"].nunique()
)

print(
    "UFs:",
    usinas["UF"].nunique()
)


print(
    "\n========== MISSING =========="
)

colunas_avaliar = [
    "UF",
    "Usina",
    "Municipio",
    "Latitude",
    "Longitude",
    "Produtos",
    "Situacao",
    "Classe_capacidade",
    "Capacidade_milho",
    "Capacidade_anidro",
    "Capacidade_hidratado",
    "Ano_construcao"
]


tabela_missing_usinas = pd.DataFrame({

    "Quantidade":
        usinas[
            colunas_avaliar
        ].isna().sum(),

    "Percentual":
        (
            usinas[
                colunas_avaliar
            ].isna().mean()
            * 100
        )
})


display(
    tabela_missing_usinas
    .round(2)
    .sort_values(
        "Percentual",
        ascending=False
    )
)


print(
    "\nDuplicatas completas:",
    usinas.duplicated().sum()
)


print(
    "Nomes de usina duplicados:",
    usinas["Usina"]
    .duplicated()
    .sum()
)

In [ ]:
# BLOCO 30.1 — DADOS POTENCIALMENTE INCORRETOS

print(
    "Latitude inválida:",
    (
        (usinas["Latitude"] < -90)
        |
        (usinas["Latitude"] > 90)
    ).sum()
)


print(
    "Longitude inválida:",
    (
        (usinas["Longitude"] < -180)
        |
        (usinas["Longitude"] > 180)
    ).sum()
)


print(
    "Capacidade anidro negativa:",
    (
        usinas["Capacidade_anidro"] < 0
    ).sum()
)


print(
    "Capacidade hidratado negativa:",
    (
        usinas["Capacidade_hidratado"] < 0
    ).sum()
)


print(
    "Capacidade milho negativa:",
    (
        usinas["Capacidade_milho"] < 0
    ).sum()
)


print(
    "Ano de construção impossível (<1800):",
    (
        usinas["Ano_construcao"] < 1800
    ).sum()
)


print(
    "Ano de construção > 2026:",
    (
        usinas["Ano_construcao"] > 2026
    ).sum()
)

In [ ]:
# BLOCO 30.2 — IDENTIFICAR USINAS RELACIONADAS AO ETANOL

usinas["Produz_etanol"] = (
    usinas["Produtos"]
    .astype(str)
    .str.contains(
        "Etanol",
        case=False,
        na=False
    )
)


usinas["Esta_operando"] = (
    usinas["Situacao"]
    .astype(str)
    .str.contains(
        "Operando",
        case=False,
        na=False
    )
)


print(
    "Usinas relacionadas ao etanol:",
    usinas["Produz_etanol"].sum()
)

print(
    "Usinas operando:",
    usinas["Esta_operando"].sum()
)


print(
    "Usinas de etanol operando:",
    (
        usinas["Produz_etanol"]
        &
        usinas["Esta_operando"]
    ).sum()
)

In [ ]:
usinas_etanol_operando = (
    usinas[
        usinas["Produz_etanol"]
        &
        usinas["Esta_operando"]
    ]
    .copy()
)


print(
    "Base utilizada na análise:",
    len(usinas_etanol_operando),
    "usinas"
)

display(
    usinas_etanol_operando.head()
)

In [ ]:
# BLOCO 31 — INDICADORES DE USINAS POR MUNICÍPIO

usinas_municipio = (
    usinas_etanol_operando
    .groupby(
        [
            "Municipio",
            "UF"
        ]
    )
    .agg(

        N_usinas=(
            "Usina",
            "nunique"
        ),

        Capacidade_anidro_total=(
            "Capacidade_anidro",
            "sum"
        ),

        Capacidade_hidratado_total=(
            "Capacidade_hidratado",
            "sum"
        ),

        Capacidade_etanol_total=(
            "Capacidade_etanol_total",
            "sum"
        ),

        Capacidade_etanol_media=(
            "Capacidade_etanol_total",
            "mean"
        ),

        Capacidade_etanol_maxima=(
            "Capacidade_etanol_total",
            "max"
        )
    )
    .reset_index()
)


display(
    usinas_municipio
    .sort_values(
        "Capacidade_etanol_total",
        ascending=False
    )
    .head(20)
)

In [ ]:
# BLOCO 31.1 — DISTRIBUIÇÃO DAS CLASSES DE CAPACIDADE

capacidade_classes = (
    usinas_etanol_operando[
        "Classe_capacidade"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)


capacidade_classes.columns = [
    "Classe_capacidade",
    "N_usinas"
]


display(
    capacidade_classes
)

In [ ]:
plt.figure(
    figsize=(10, 6)
)

sns.barplot(
    data=capacidade_classes,
    x="N_usinas",
    y="Classe_capacidade"
)

plt.title(
    "Distribuição das usinas de etanol por classe de capacidade"
)

plt.xlabel(
    "Número de usinas"
)

plt.ylabel(
    "Classe de capacidade"
)

plt.tight_layout()
plt.show()

In [ ]:
# BLOCO 32 — EDA QUANTITATIVA DAS USINAS

variaveis_usinas = [
    "N_usinas",
    "Capacidade_anidro_total",
    "Capacidade_hidratado_total",
    "Capacidade_etanol_total",
    "Capacidade_etanol_media",
    "Capacidade_etanol_maxima"
]

# 1. OUTLIERS

print(
    "========== OUTLIERS =========="
)

display(
    tabela_outliers(
        usinas_municipio,
        variaveis_usinas
    ).round(2)
)

# 2. VARIAÇÃO RELATIVA

print(
    "\n========== VARIAÇÃO RELATIVA =========="
)

display(
    tabela_variacao(
        usinas_municipio,
        variaveis_usinas
    ).round(2)
)

# 3. NORMALIDADE

print(
    "\n========== NORMALIDADE =========="
)

display(
    tabela_normalidade(
        usinas_municipio,
        variaveis_usinas
    )
)

# 4. CORRELAÇÃO

print(
    "\n========== CORRELAÇÃO =========="
)

corr_usinas = (
    usinas_municipio[
        variaveis_usinas
    ]
    .corr(
        method="pearson"
    )
)

display(
    corr_usinas.round(3)
)

In [ ]:
# BLOCO 32.1 — HEATMAP DAS USINAS

plt.figure(
    figsize=(9, 7)
)

sns.heatmap(
    corr_usinas,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title(
    "Correlação dos indicadores municipais das usinas"
)

plt.tight_layout()
plt.show()

In [ ]:
# BLOCO 33 — DISTRIBUIÇÕES DOS INDICADORES DAS USINAS

for var in variaveis_usinas:

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11, 4)
    )


    sns.histplot(
        data=usinas_municipio,
        x=var,
        kde=True,
        ax=axes[0]
    )


    sns.boxplot(
        data=usinas_municipio,
        x=var,
        ax=axes[1]
    )


    axes[0].set_title(
        f"Histograma — {var}"
    )


    axes[1].set_title(
        f"Boxplot — {var}"
    )


    plt.tight_layout()
    plt.show()

In [ ]:
# BLOCO 33.1 — INDICADORES POR UF

usinas_estado = (
    usinas_etanol_operando
    .groupby("UF")
    .agg(

        N_usinas=(
            "Usina",
            "nunique"
        ),

        Capacidade_anidro_total=(
            "Capacidade_anidro",
            "sum"
        ),

        Capacidade_hidratado_total=(
            "Capacidade_hidratado",
            "sum"
        ),

        Capacidade_etanol_total=(
            "Capacidade_etanol_total",
            "sum"
        )
    )
    .reset_index()
)


display(
    usinas_estado
    .sort_values(
        "Capacidade_etanol_total",
        ascending=False
    )
)

In [ ]:
plt.figure(
    figsize=(10, 6)
)

dados_plot = (
    usinas_estado
    .sort_values(
        "Capacidade_etanol_total",
        ascending=False
    )
    .head(15)
)


sns.barplot(
    data=dados_plot,
    x="Capacidade_etanol_total",
    y="UF"
)

#4. DISTRIBUIDORAS

In [ ]:
# BLOCO 34 — DIAGNÓSTICO DAS DISTRIBUIDORAS

print("FORMATO:")
print(df_distribuidoras.shape)

print("\nCOLUNAS:")
print(df_distribuidoras.columns.tolist())

print("\nPRIMEIRAS LINHAS:")
display(df_distribuidoras.head(10))

In [ ]:
# BLOCO 35 — CORRIGIR CABEÇALHO DAS DISTRIBUIDORAS

# A linha de índice 3 contém o cabeçalho verdadeiro
novo_cabecalho = df_distribuidoras.iloc[3].copy()

# Os dados começam na linha seguinte
distribuidoras = (
    df_distribuidoras
    .iloc[4:]
    .copy()
)

# Aplicar cabeçalho correto
distribuidoras.columns = novo_cabecalho

# Reiniciar índice
distribuidoras = (
    distribuidoras
    .reset_index(drop=True)
)

print("Formato após correção:")
print(distribuidoras.shape)

print("\nColunas corrigidas:")
print(distribuidoras.columns.tolist())

display(
    distribuidoras.head(10)
)

In [ ]:
# BLOCO 35.1 — NORMALIZAR NOMES DAS COLUNAS

distribuidoras.columns = [
    normalizar_coluna(coluna)
    for coluna in distribuidoras.columns
]

print("Colunas normalizadas:")

for coluna in distribuidoras.columns:
    print("-", coluna)

display(
    distribuidoras.head()
)

In [ ]:
# BLOCO 36 — TRATAMENTO DAS DISTRIBUIDORAS

distribuidoras = distribuidoras.rename(
    columns={
        "codigo_agente":
            "Codigo_agente",

        "codigo_agente_i_simp":
            "Codigo_iSIMP",

        "cnpj":
            "CNPJ",

        "nome_reduzido":
            "Nome",

        "razao_social":
            "Razao_social",

        "endereco_da_matriz":
            "Endereco",

        "bairro":
            "Bairro",

        "municipio":
            "Municipio",

        "uf":
            "UF",

        "cep":
            "CEP",

        "situacao":
            "Situacao",

        "inicio_da_situacao":
            "Inicio_situacao",

        "data_publicacao":
            "Data_publicacao"
    }
).copy()

# LIMPEZA DE TEXTO

colunas_texto = [
    "CNPJ",
    "Nome",
    "Razao_social",
    "Municipio",
    "UF",
    "Situacao"
]

for coluna in colunas_texto:

    if coluna in distribuidoras.columns:

        distribuidoras[coluna] = (
            distribuidoras[coluna]
            .astype("string")
            .str.strip()
        )

# DATAS

distribuidoras["Inicio_situacao"] = pd.to_datetime(
    distribuidoras["Inicio_situacao"],
    dayfirst=True,
    errors="coerce"
)

distribuidoras["Data_publicacao"] = pd.to_datetime(
    distribuidoras["Data_publicacao"],
    dayfirst=True,
    errors="coerce"
)


print(
    "Formato final:",
    distribuidoras.shape
)

display(
    distribuidoras.head(10)
)

In [ ]:
# BLOCO 37 — QUALIDADE DA BASE DE DISTRIBUIDORAS

print(
    "Registros:",
    len(distribuidoras)
)

print(
    "CNPJs únicos:",
    distribuidoras["CNPJ"].nunique()
)

print(
    "Municípios:",
    distribuidoras["Municipio"].nunique()
)

print(
    "UFs:",
    distribuidoras["UF"].nunique()
)

print(
    "Duplicatas completas:",
    distribuidoras.duplicated().sum()
)

print(
    "CNPJs duplicados:",
    distribuidoras["CNPJ"]
    .duplicated()
    .sum()
)


print(
    "\n========== MISSING =========="
)

colunas_avaliar_dist = [
    "CNPJ",
    "Nome",
    "Razao_social",
    "Municipio",
    "UF",
    "Situacao",
    "Inicio_situacao",
    "Data_publicacao"
]


missing_distribuidoras = pd.DataFrame({

    "Quantidade":
        distribuidoras[
            colunas_avaliar_dist
        ].isna().sum(),

    "Percentual":
        (
            distribuidoras[
                colunas_avaliar_dist
            ].isna().mean()
            * 100
        )
})


display(
    missing_distribuidoras
    .round(2)
    .sort_values(
        "Percentual",
        ascending=False
    )
)

In [ ]:
# BLOCO 37.1 — SITUAÇÃO CADASTRAL

situacoes_distribuidoras = (
    distribuidoras["Situacao"]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

situacoes_distribuidoras.columns = [
    "Situacao",
    "Quantidade"
]

display(
    situacoes_distribuidoras
)

In [ ]:
# BLOCO 37.2 — DISTRIBUIDORAS ATUALMENTE AUTORIZADAS

distribuidoras["Esta_autorizada"] = (
    distribuidoras["Situacao"]
    .astype(str)
    .str.upper()
    .str.strip()
    .eq("AUTORIZADA")
)


distribuidoras_ativas = (
    distribuidoras[
        distribuidoras["Esta_autorizada"]
    ]
    .copy()
)


print(
    "Total de registros:",
    len(distribuidoras)
)

print(
    "Distribuidoras autorizadas:",
    len(distribuidoras_ativas)
)

print(
    "CNPJs autorizados únicos:",
    distribuidoras_ativas[
        "CNPJ"
    ].nunique()
)

print(
    "Municípios com distribuidora autorizada:",
    distribuidoras_ativas[
        "Municipio"
    ].nunique()
)

print(
    "UFs:",
    distribuidoras_ativas[
        "UF"
    ].nunique()
)

In [ ]:
# BLOCO 38 — DISTRIBUIDORAS POR MUNICÍPIO

distribuidoras_municipio = (
    distribuidoras_ativas
    .groupby(
        [
            "Municipio",
            "UF"
        ]
    )
    .agg(

        N_distribuidoras=(
            "CNPJ",
            "nunique"
        )
    )
    .reset_index()
)


display(
    distribuidoras_municipio
    .sort_values(
        "N_distribuidoras",
        ascending=False
    )
    .head(25)
)

In [ ]:
# BLOCO 38.1 — DISTRIBUIDORAS POR ESTADO

distribuidoras_estado = (
    distribuidoras_ativas
    .groupby("UF")
    .agg(

        N_distribuidoras=(
            "CNPJ",
            "nunique"
        ),

        N_municipios=(
            "Municipio",
            "nunique"
        )
    )
    .reset_index()
)


display(
    distribuidoras_estado
    .sort_values(
        "N_distribuidoras",
        ascending=False
    )
)

In [ ]:
# BLOCO 39 — EDA DAS DISTRIBUIDORAS

variaveis_distribuidoras = [
    "N_distribuidoras"
]

# OUTLIERS

print(
    "========== OUTLIERS =========="
)

display(
    tabela_outliers(
        distribuidoras_municipio,
        variaveis_distribuidoras
    ).round(2)
)

# VARIAÇÃO RELATIVA

print(
    "\n========== VARIAÇÃO RELATIVA =========="
)

display(
    tabela_variacao(
        distribuidoras_municipio,
        variaveis_distribuidoras
    ).round(2)
)

# NORMALIDADE

print(
    "\n========== NORMALIDADE =========="
)

display(
    tabela_normalidade(
        distribuidoras_municipio,
        variaveis_distribuidoras
    )
)

In [ ]:
# BLOCO 39.1 — DISTRIBUIÇÃO ESPACIAL DAS DISTRIBUIDORAS

plt.figure(
    figsize=(9, 4)
)

sns.histplot(
    data=distribuidoras_municipio,
    x="N_distribuidoras",
    bins=20,
    kde=True
)

plt.title(
    "Distribuição do número de distribuidoras autorizadas por município"
)

plt.xlabel(
    "Número de distribuidoras"
)

plt.ylabel(
    "Número de municípios"
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(9, 3)
)

sns.boxplot(
    data=distribuidoras_municipio,
    x="N_distribuidoras"
)

plt.title(
    "Boxplot — distribuidoras autorizadas por município"
)

plt.xlabel(
    "Número de distribuidoras"
)

plt.tight_layout()
plt.show()

In [ ]:
# BLOCO 39.2 — TOP MUNICÍPIOS

top_distribuidoras = (
    distribuidoras_municipio
    .sort_values(
        "N_distribuidoras",
        ascending=False
    )
    .head(20)
)


display(
    top_distribuidoras
)

In [ ]:
plt.figure(
    figsize=(10, 7)
)

sns.barplot(
    data=top_distribuidoras,
    x="N_distribuidoras",
    y="Municipio"
)

plt.title(
    "Municípios com maior número de distribuidoras autorizadas"
)

plt.xlabel(
    "Número de distribuidoras"
)

plt.ylabel(
    "Município"
)

plt.tight_layout()
plt.show()

#5. POSTOS DE COMBUSTÍVEIS

In [ ]:
# BLOCO 40 — DIAGNÓSTICO DOS POSTOS

print("FORMATO:")
print(df_postos.shape)

print("\nCOLUNAS:")
print(df_postos.columns.tolist())

print("\nPRIMEIRAS LINHAS:")
display(df_postos.head(10))

In [ ]:
# BLOCO 41 — TRATAMENTO DA BASE DE POSTOS

postos = df_postos.rename(
    columns={
        "codigoisimp":
            "Codigo_iSIMP",

        "autorizacao":
            "Autorizacao",

        "datapublicacao":
            "Data_publicacao",

        "razaosocial":
            "Razao_social",

        "cnpj":
            "CNPJ",

        "endereco":
            "Endereco",

        "complemento":
            "Complemento",

        "bairro":
            "Bairro",

        "cep":
            "CEP",

        "uf":
            "UF",

        "municipio":
            "Municipio",

        "bandeira":
            "Bandeira",

        "datavinculacao":
            "Data_vinculacao"
    }
).copy()

# CNPJ COMO TEXTO

postos["CNPJ"] = (
    postos["CNPJ"]
    .astype("string")
    .str.replace(
        r"\D",
        "",
        regex=True
    )
    .str.zfill(14)
)

# LIMPEZA DAS VARIÁVEIS DE TEXTO

colunas_texto_postos = [
    "Razao_social",
    "Endereco",
    "Bairro",
    "UF",
    "Municipio",
    "Bandeira"
]

for coluna in colunas_texto_postos:

    postos[coluna] = (
        postos[coluna]
        .astype("string")
        .str.strip()
    )

# DATAS

postos["Data_publicacao"] = pd.to_datetime(
    postos["Data_publicacao"],
    dayfirst=True,
    errors="coerce"
)

postos["Data_vinculacao"] = pd.to_datetime(
    postos["Data_vinculacao"],
    dayfirst=True,
    errors="coerce"
)


print(
    "Formato final:",
    postos.shape
)

display(
    postos.head(10)
)

In [ ]:
# BLOCO 42 — QUALIDADE DA BASE DE POSTOS

print(
    "Registros:",
    len(postos)
)

print(
    "CNPJs únicos:",
    postos["CNPJ"].nunique()
)

print(
    "Municípios:",
    postos["Municipio"].nunique()
)

print(
    "UFs:",
    postos["UF"].nunique()
)

print(
    "Bandeiras diferentes:",
    postos["Bandeira"].nunique()
)

print(
    "Duplicatas completas:",
    postos.duplicated().sum()
)

print(
    "CNPJs duplicados:",
    postos["CNPJ"]
    .duplicated()
    .sum()
)

In [ ]:
# BLOCO 42.1 — MISSING DA BASE DE POSTOS

colunas_avaliar_postos = [
    "Codigo_iSIMP",
    "Autorizacao",
    "Data_publicacao",
    "Razao_social",
    "CNPJ",
    "Endereco",
    "Bairro",
    "UF",
    "Municipio",
    "Bandeira",
    "Data_vinculacao"
]


missing_postos = pd.DataFrame({

    "Quantidade":
        postos[
            colunas_avaliar_postos
        ].isna().sum(),

    "Percentual":
        (
            postos[
                colunas_avaliar_postos
            ].isna().mean()
            * 100
        )
})


display(
    missing_postos
    .round(2)
    .sort_values(
        "Percentual",
        ascending=False
    )
)

In [ ]:
# BLOCO 42.2 — DADOS POTENCIALMENTE INCORRETOS

ufs_validas = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO"
]


print(
    "CNPJ com tamanho diferente de 14 dígitos:",
    (
        postos["CNPJ"]
        .str.len()
        != 14
    ).sum()
)


print(
    "UF inválida:",
    (
        ~postos["UF"]
        .isin(ufs_validas)
    ).sum()
)


print(
    "Município ausente:",
    postos["Municipio"]
    .isna()
    .sum()
)


print(
    "Data publicação inválida:",
    postos["Data_publicacao"]
    .isna()
    .sum()
)


print(
    "Data vinculação inválida:",
    postos["Data_vinculacao"]
    .isna()
    .sum()
)

In [ ]:
# BLOCO 43 — POSTOS POR MUNICÍPIO

postos_municipio = (
    postos
    .groupby(
        [
            "Municipio",
            "UF"
        ]
    )
    .agg(

        N_postos=(
            "CNPJ",
            "nunique"
        ),

        N_bandeiras=(
            "Bandeira",
            "nunique"
        ),

        Primeira_autorizacao=(
            "Data_publicacao",
            "min"
        ),

        Ultima_autorizacao=(
            "Data_publicacao",
            "max"
        )
    )
    .reset_index()
)


display(
    postos_municipio
    .sort_values(
        "N_postos",
        ascending=False
    )
    .head(25)
)

In [ ]:
# BLOCO 43.1 — BANDEIRA BRANCA POR MUNICÍPIO

postos["Bandeira_branca"] = (
    postos["Bandeira"]
    .astype(str)
    .str.upper()
    .str.contains(
        "BANDEIRA BRANCA",
        na=False
    )
)


bandeira_branca_municipio = (
    postos
    .groupby(
        [
            "Municipio",
            "UF"
        ]
    )
    .agg(

        Postos_total=(
            "CNPJ",
            "nunique"
        ),

        Postos_bandeira_branca=(
            "Bandeira_branca",
            "sum"
        )
    )
    .reset_index()
)


bandeira_branca_municipio[
    "Pct_bandeira_branca"
] = (
    bandeira_branca_municipio[
        "Postos_bandeira_branca"
    ]
    /
    bandeira_branca_municipio[
        "Postos_total"
    ]
) * 100


postos_municipio = (
    postos_municipio
    .merge(
        bandeira_branca_municipio[
            [
                "Municipio",
                "UF",
                "Pct_bandeira_branca"
            ]
        ],
        on=[
            "Municipio",
            "UF"
        ],
        how="left"
    )
)


display(
    postos_municipio.head()
)

In [ ]:
# BLOCO 44 — EDA DOS POSTOS

variaveis_postos = [
    "N_postos",
    "N_bandeiras",
    "Pct_bandeira_branca"
]

# 1. OUTLIERS

print(
    "========== OUTLIERS =========="
)

display(
    tabela_outliers(
        postos_municipio,
        variaveis_postos
    ).round(2)
)

# 2. VARIAÇÃO RELATIVA

print(
    "\n========== VARIAÇÃO RELATIVA =========="
)

display(
    tabela_variacao(
        postos_municipio,
        variaveis_postos
    ).round(2)
)

# 3. NORMALIDADE

print(
    "\n========== NORMALIDADE =========="
)

display(
    tabela_normalidade(
        postos_municipio,
        variaveis_postos
    )
)

# 4. CORRELAÇÃO

print(
    "\n========== CORRELAÇÃO =========="
)

corr_postos = (
    postos_municipio[
        variaveis_postos
    ]
    .corr(
        method="pearson"
    )
)

display(
    corr_postos.round(3)
)

In [ ]:
# BLOCO 44.1 — HEATMAP DOS POSTOS

plt.figure(
    figsize=(7, 6)
)

sns.heatmap(
    corr_postos,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title(
    "Correlação dos indicadores municipais de postos"
)

plt.tight_layout()
plt.show()

In [ ]:
# BLOCO 45 — DISTRIBUIÇÕES DOS POSTOS

for var in variaveis_postos:

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11, 4)
    )


    sns.histplot(
        data=postos_municipio,
        x=var,
        kde=True,
        ax=axes[0]
    )


    sns.boxplot(
        data=postos_municipio,
        x=var,
        ax=axes[1]
    )


    axes[0].set_title(
        f"Histograma — {var}"
    )


    axes[1].set_title(
        f"Boxplot — {var}"
    )


    plt.tight_layout()
    plt.show()

In [ ]:
#  BLOCO 45.1 — TOP MUNICÍPIOS POR NÚMERO DE POSTOS

top_postos = (
    postos_municipio
    .sort_values(
        "N_postos",
        ascending=False
    )
    .head(20)
)


display(
    top_postos
)

In [ ]:
plt.figure(
    figsize=(10, 7)
)

sns.barplot(
    data=top_postos,
    x="N_postos",
    y="Municipio"
)

plt.title(
    "Municípios com maior número de postos"
)

plt.xlabel(
    "Número de postos"
)

plt.ylabel(
    "Município"
)

plt.tight_layout()
plt.show()

In [ ]:
# BLOCO 45.2 — POSTOS POR ESTADO

postos_estado = (
    postos
    .groupby("UF")
    .agg(

        N_postos=(
            "CNPJ",
            "nunique"
        ),

        N_municipios=(
            "Municipio",
            "nunique"
        ),

        N_bandeiras=(
            "Bandeira",
            "nunique"
        )
    )
    .reset_index()
)


display(
    postos_estado
    .sort_values(
        "N_postos",
        ascending=False
    )
)

In [ ]:
# BLOCO 45.3 — PRINCIPAIS BANDEIRAS

bandeiras = (
    postos["Bandeira"]
    .value_counts()
    .reset_index()
)

bandeiras.columns = [
    "Bandeira",
    "N_postos"
]


display(
    bandeiras.head(20)
)

In [ ]:
# BLOCO 46 — COMPARAÇÃO DAS CINCO FONTES

comparacao_fontes = pd.DataFrame({

    "Fonte": [
        "Produção de cana",
        "Produção de etanol",
        "Usinas",
        "Distribuidoras",
        "Postos"
    ],

    "Escala_espacial": [
        "Município",
        "Estado",
        "Município / ponto",
        "Município da matriz",
        "Município / estabelecimento"
    ],

    "Escala_temporal": [
        "Anual 2021–2024",
        "Anual 2021–2026",
        "Fotografia cadastral",
        "Fotografia 2026",
        "Fotografia cadastral"
    ],

    "Lado_da_rota": [
        "Origem",
        "Origem",
        "Origem",
        "Destino",
        "Destino"
    ],

    "Significado_logistico": [
        "Oferta de matéria-prima",
        "Oferta estadual de etanol",
        "Infraestrutura produtiva de etanol",
        "Estrutura de distribuição",
        "Demanda varejista potencial"
    ]
})


display(
    comparacao_fontes
)

In [ ]:
# BLOCO 47 — VARIÁVEIS CANDIDATAS AO ESTUDO DE FRETES

variaveis_candidatas = pd.DataFrame({

    "Fonte": [
        "Cana",
        "Etanol estadual",
        "Usinas",
        "Usinas",
        "Distribuidoras",
        "Postos"
    ],

    "Variavel": [
        "Media_cana",
        "Media_etanol",
        "N_usinas",
        "Capacidade_etanol_total",
        "N_distribuidoras",
        "N_postos"
    ],

    "Associacao": [
        "Município de origem",
        "UF de origem",
        "Município de origem",
        "Município de origem",
        "Município de destino",
        "Município de destino"
    ]
})


display(
    variaveis_candidatas
)

In [ ]:
# BLOCO 48 — ADEQUAÇÃO METODOLÓGICA AO ESTUDO DE FRETES
#
# 1 = baixa adequação
# 2 = baixa-média
# 3 = média
# 4 = alta
# 5 = muito alta
#
# Estas notas são uma avaliação metodológica transparente,
# não um resultado estatístico.
# ============================================================

avaliacao_fontes = pd.DataFrame({

    "Fonte": [
        "Cana municipal",
        "Etanol estadual",
        "Usinas",
        "Distribuidoras",
        "Postos"
    ],

    "Resolucao_espacial": [
        5,
        2,
        5,
        4,
        5
    ],

    "Compatibilidade_temporal": [
        3,
        3,
        2,
        2,
        2
    ],

    "Relacao_direta_com_frete_etanol": [
        4,
        4,
        5,
        4,
        3
    ],

    "Interpretabilidade_logistica": [
        4,
        4,
        5,
        4,
        4
    ],

    "Adequacao_a_origem_destino": [
        5,
        2,
        5,
        3,
        5
    ]
})


avaliacao_fontes["Media_metodologica"] = (
    avaliacao_fontes[
        [
            "Resolucao_espacial",
            "Compatibilidade_temporal",
            "Relacao_direta_com_frete_etanol",
            "Interpretabilidade_logistica",
            "Adequacao_a_origem_destino"
        ]
    ]
    .mean(axis=1)
)


avaliacao_fontes = (
    avaliacao_fontes
    .sort_values(
        "Media_metodologica",
        ascending=False
    )
)


display(
    avaliacao_fontes.round(2)
)

In [ ]:
# BLOCO 49 — CONCLUSÃO METODOLÓGICA

print("""
============================================================
CONCLUSÃO DA ANÁLISE EXPLORATÓRIA DAS FONTES EXTERNAS
============================================================

1. PRODUÇÃO MUNICIPAL DE CANA
------------------------------------------------------------
Vantagens:
- nível municipal;
- ligação direta com a oferta agrícola;
- fácil associação à origem da rota.

Limitações:
- frequência anual;
- série disponível somente até 2024;
- cana pode ser destinada tanto ao açúcar quanto ao etanol.

Uso recomendado:
- caracterização produtiva da origem;
- variável complementar de robustez.


2. PRODUÇÃO ESTADUAL DE ETANOL
------------------------------------------------------------
Vantagens:
- representa diretamente a oferta de etanol;
- série temporal de 2021 a 2026.

Limitações:
- nível estadual;
- frequência anual;
- várias rotas municipais recebem o mesmo valor estadual.

Uso recomendado:
- variável contextual da oferta regional.


3. USINAS DE ETANOL
------------------------------------------------------------
Vantagens:
- localização municipal;
- coordenadas geográficas;
- identifica infraestrutura efetivamente produtiva;
- permite selecionar usinas operando;
- possui capacidade de anidro e hidratado;
- está diretamente relacionada à origem física do álcool.

Limitações:
- essencialmente fotografia da infraestrutura;
- capacidade não equivale a produção efetiva;
- capacidade de moagem está disponível em classes.

Uso recomendado:
- PRINCIPAL variável externa candidata ao estudo de fretes.


4. DISTRIBUIDORAS
------------------------------------------------------------
Vantagens:
- ligação com a etapa de distribuição;
- localização municipal;
- situação cadastral disponível.

Limitações:
- arquivo representa principalmente endereço da matriz;
- não necessariamente localização da base física;
- fotografia de 2026.

Uso recomendado:
- caracterização complementar do destino;
- não utilizar como principal variável sem validar a
  localização das bases operacionais.


5. POSTOS DE COMBUSTÍVEIS
------------------------------------------------------------
Vantagens:
- excelente resolução municipal;
- grande cobertura nacional;
- representa a estrutura varejista da demanda;
- permite medir número de postos e diversidade de bandeiras.

Limitações:
- não representa diretamente o destino imediato do frete
  primário de etanol;
- fotografia cadastral, e não série mensal histórica.

Uso recomendado:
- caracterização da demanda potencial no destino;
- interpretação posterior dos clusters.


============================================================
DADO PRIORITÁRIO
============================================================

Entre as cinco fontes, LOCALIZAÇÃO E CAPACIDADE DAS USINAS
é a candidata mais adequada para complementar diretamente
a análise de agrupamentos do preço de frete de álcool.

Indicadores prioritários:

- N_usinas na origem;
- Capacidade_anidro_total na origem;
- Capacidade_hidratado_total na origem;
- Capacidade_etanol_total na origem.

Produção municipal de cana é uma excelente segunda fonte.

Distribuidoras e postos são mais úteis para caracterizar o
lado do destino.

Produção estadual de etanol deve ser tratada principalmente
como variável contextual regional.
============================================================
""")